In [1]:
# Import libraries for data preprocessing
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

# Import libraries for ML
import torch
import torch.nn as nn
from torch.optim import SGD
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

c) We have provided a dataset from taxi trips taken in NYC in taxi_trips.parquet. It contains 8
total columns, 7 features and 1 target (trip duration, in minutes). The 7 features (pickup/dropoff
coordinates and time-of-day, day-of-week and day-of-year) have already been standardized, so do
not expect them to look like raw latitudes and longitudes. Define a multi-layer perceptron in
PyTorch or JAX that outputs a conditional mean and standard deviation, and fit this on the taxi
trip dataset. Make sure to follow best practices: define a baseline (or multiple) baseline models
that you think your model should beat, split your dataset into training/validation/test sets, etc.

In [2]:
# Load dataset
df = pd.read_parquet("taxi_trips.parquet")
df.head()

,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,trip_duration,min_of_day,day_of_week,day_of_year
187173,0.270744,1.425051,0.466831,0.776192,7.965142,-0.942057,-1.047619,1.180855
179655,0.085681,0.306079,-0.226274,0.124111,10.680277,0.778703,-1.047619,-0.311290
120048,-0.555533,-0.934806,-0.022489,-0.548561,5.716184,-1.183016,-1.047619,1.316504
62587,0.069718,0.035804,0.258900,0.317284,4.266709,-0.908008,0.481657,0.018144
212063,-0.234482,0.074616,0.115836,0.258310,11.629685,1.066806,-0.537860,0.521985


In [3]:
# Split to X and Y sets
X = df.drop(columns=["trip_duration"])
y = df["trip_duration"]

print(X.shape)
print(y.shape)

(523554, 7)
(523554,)


In [ ]:
# Split into training and test sets
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

# Split into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.15 / 0.85, random_state=42)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)
print(y_train.shape)
print(y_val.shape)
print(y_test.shape)

(366487, 7)
(78533, 7)
(78534, 7)
(366487,)
(78533,)
(78534,)


In [5]:
# Convert features to tensors
X_train_tensor = torch.tensor(X_train.to_numpy(), dtype=torch.float32).to(device='xpu')
X_val_tensor = torch.tensor(X_val.to_numpy(), dtype=torch.float32).to(device='xpu')
X_test_tensor = torch.tensor(X_test.to_numpy(), dtype=torch.float32).to(device='xpu')

# Convert targets to tensors
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.float32).to(device='xpu')
y_val_tensor = torch.tensor(y_val.to_numpy(), dtype=torch.float32).to(device='xpu')
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype=torch.float32).to(device='xpu')

# Create datasets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

In [6]:
# Fit baseline linear regression for conditional means
lr = LinearRegression()
lr.fit(X_train, y_train)

# Predict conditional means
lr_train_pred = lr.predict(X_train)
lr_val_pred = lr.predict(X_val)
lr_test_pred = lr.predict(X_test)

# Estimate baseline constant conditional SD using training residuals
lr_sd = np.sqrt(np.mean((y_train.to_numpy() - lr_train_pred) ** 2))

# Predict the same SD for every observation
lr_test_sd = np.full(len(y_test), lr_sd)

# Calculate test metrics
lr_test_rmse = np.sqrt(mean_squared_error(y_test, lr_test_pred))
lr_test_nll = np.mean(
    0.5 * np.log(2 * np.pi * lr_test_sd**2) 
    + (y_test.to_numpy() - lr_test_pred)**2 / (2 * lr_test_sd**2)
)

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim = 7, hidden_dim = 128, device: str = "xpu"):
        super().__init__()

        # Use GPU
        self.device = device

        # Architecture
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim,2)
        )

    def forward(self, x):
        output = self.net(x)

        mu = output[:, 0]
        sigma = torch.nn.functional.softplus(output[:, 1])
        return mu, sigma

# Set up model and optimizer
model = MLP().to(device='xpu')
optimizer = SGD(model.parameters(), lr=0.001)

# Set up loss function
def normal_nll(y_true, mu, sigma):
    return torch.mean(
        0.5 * torch.log(2 * torch.pi * sigma**2)
        + (y_true - mu)**2 / (2 * sigma**2)
    )

In [8]:
epochs = 100

for epoch in range(epochs):
    # Training MLP
    model.train()
    train_losses = []

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()

        # Calculate loss
        mu, sigma = model(X_batch)
        loss_train = normal_nll(y_batch, mu, sigma)

        # Update weights
        loss_train.backward()
        optimizer.step()

        train_losses.append(loss_train.item())

    # Validation
    model.eval()
    val_losses = []

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            # Calculate validation loss
            mu, sigma = model(X_batch)
            loss_val = normal_nll(y_batch, mu, sigma)

            val_losses.append(loss_val.item())

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch+1}: "
            f"Train Loss = {np.mean(train_losses):.4f}, "
            f"Validation Loss = {np.mean(val_losses):.4f}"
        )

Epoch 10: Train Loss = 3.1034, Validation Loss = 3.0823
Epoch 20: Train Loss = 2.8523, Validation Loss = 2.8443
Epoch 30: Train Loss = 2.7873, Validation Loss = 2.7845
Epoch 40: Train Loss = 2.7603, Validation Loss = 2.7590
Epoch 50: Train Loss = 2.7419, Validation Loss = 2.7421
Epoch 60: Train Loss = 2.7277, Validation Loss = 2.7301
Epoch 70: Train Loss = 2.7158, Validation Loss = 2.7202
Epoch 80: Train Loss = 2.7056, Validation Loss = 2.7117
Epoch 90: Train Loss = 2.6960, Validation Loss = 2.7035
Epoch 100: Train Loss = 2.6868, Validation Loss = 2.6987


In [9]:
# Evaluate on test set
model.eval()

mlp_means = []
mlp_sds = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        mu, sigma = model(X_batch)

        mlp_means.extend(mu.cpu().numpy())
        mlp_sds.extend(sigma.cpu().numpy())


mlp_means = np.array(mlp_means)
mlp_sds = np.array(mlp_sds)


# Calculate test RMSE
mlp_test_rmse = np.sqrt(mean_squared_error(y_test, mlp_means))


# Calculate test NLL
mlp_test_nll = np.mean(
    0.5 * np.log(2 * np.pi * mlp_sds**2)
    + (y_test.to_numpy() - mlp_means)**2 / (2 * mlp_sds**2)
)

In [10]:
print("Linear Regression Baseline")
print("Conditional SD:", lr_sd)
print("Test RMSE:", lr_test_rmse) # want to minimize
print("Test NLL:", lr_test_nll) # want to minimize

print("\nGaussian MLP")
print("Test RMSE:", mlp_test_rmse) # want to minimize
print("Test NLL:", mlp_test_nll) # want to minimize

Linear Regression Baseline
Conditional SD: 7.548108
Test RMSE: 7.552151495965741
Test NLL: 3.4407713

Gaussian MLP
Test RMSE: 4.398986673711704
Test NLL: 2.8034852


e) Estimate these probabilities on your held-out test set and evaluate them. Report both a proper
scoring rule for the predicted probabilities (e.g. the Brier score or the log-loss) and the classification
accuracy obtained by thresholding the probability at 0.5.

In [11]:
# Import libraries
from scipy.stats import norm
from sklearn.metrics import brier_score_loss, accuracy_score

In [12]:
# Evaluate on test set
model.eval()
probabilities = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        mu, sigma = model(X_batch)

        # Calculate probabilities using Normal cdf
        batch_probs = norm.cdf((45 - mu.cpu().numpy()) / sigma.cpu().numpy())

        probabilities.extend(batch_probs)

probabilities = np.array(probabilities)
probabilities.shape


(78534,)

In [13]:
# Convert to binary classification
y_target = (y_test.to_numpy() < 45).astype(int)

# Calculate Brier score loss
test_brier = brier_score_loss(y_target, probabilities)

print("Brier Score:", test_brier) # want to minimize

Brier Score: 0.0037527918334624303


In [14]:
# Converting probabilities to classes with threshold 0.5
y_pred = (probabilities >= 0.5).astype(int)

# Calculate accuracy
accuracy = accuracy_score(y_target, y_pred)

print("Classification Accuracy:", accuracy) # want to maximize

Classification Accuracy: 0.9953395981358393
